# Exercise: Molecular Crystal Optimization

In this exercise we will be optimizing the atomic positions and lattice of molecular crystals.  Specifically, we have a structure from the CSD (identifier: COWCAS)

![title](cowcas.png)

and a co-crystal structure for a knoevenagel reaction between barbituric acid and vanillin generated by Clari (https://github.com/the-matter-lab/clari).

**Kernel:** MACE. Run cells from top to bottom in a fresh kernel. 

**Learning goals:** distinguish atomic and cell relaxation; inspect stress and volume; recognize step-limited results.

**Working pattern:** predict a result, run the calculation, inspect the geometry and convergence,
then explain the result to a partner. Energy is reported in eV, length in angstrom, and force in eV/angstrom.


In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0,str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view

from demo_tools.crystal_view import repair_fragmented_molecules
from demo_tools.helpers import start_exercise

_, OUTPUT = start_exercise()

Python: /home/nchopper/anaconda3/envs/mace_demo/bin/python
Inputs: /home/nchopper/mlip-demo/workshop_demo/MolecularCrystals
New results: /home/nchopper/mlip-demo/workshop_demo/MolecularCrystals/results/20260923-163153-2c0ef3


## 1. Inspect the starting structures
`COWCAS.cif` is the supplied CSD structure; `my_cocrystal.cif` is the supplied generated candidate
for barbituric acid and vanillin. Inspect one at a time. The display helper reconnects molecular
fragments across periodic boundaries for visualization; calculations use the original periodic structure.

Choose COWCAS for the first pass. Comparing total energies of crystals with different compositions
or numbers of atoms does not establish their relative stability.

**Input inspection note:** ASE emitted symmetry/equivalent-site warnings for COWCAS during input checks. Inspect the expanded atom count, cell, and connectivity with the instructor before treating it as a validated reference. The checked reader produced 172 atoms for COWCAS and 64 for the generated co-crystal; matching these counts alone does not establish correctness.


In [6]:
cowcas = read('./COWCAS.cif')
my_cocrystal = read('./my_cocrystal.cif')

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'orthorhombic' is not interpreted for space group Spacegroup(36, setting=1). This may result in wrong setting!
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 3 and 13 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 5 and 14 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 4 and 15 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/spacegroup/spacegroup.py:484: UserWarning: scaled_positions 0 and 16 are equivalent
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-

In [9]:
view(repair_fragmented_molecules(cowcas, centroid_position_frac=[0.5,0.5,0.5]),viewer='x3d')

In [12]:
view(my_cocrystal,viewer='x3d')

In [14]:
from mace.calculators import MACECalculator
DEVICE = 'cpu'  # Only select cuda inside a GPU allocation with a compatible environment.
calculator = MACECalculator(model_paths='../models/2023-12-10-mace-128-L0_energy_epoch-249.model',
                            device=DEVICE, default_dtype='float64')


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


## 2. Compare two optimization choices
Both calculations start from independent copies of the same geometry. Fixed-cell relaxation moves
atoms only; `FrechetCellFilter` exposes cell degrees of freedom to the optimizer as well.
The cell filter couples lattice and atomic relaxation—it is not a second independent calculation of stress.

These bounded runs may stop before convergence. Report that outcome rather than calling every final
structure optimized. The maximum force reported below is atomic; the cell-filter convergence criterion
also includes cell-related components. Inspect stress separately.


In [ ]:
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE
fixed, variable = initial.copy(), initial.copy()
fixed.calc = calculator
variable.calc = calculator
initial_energy = initial.get_potential_energy()
MAX_STEPS = 100
fixed_ok = relax(fixed, FIRE, fmax=0.05, steps=MAX_STEPS, logfile=str(OUTPUT / 'fixed.log'))
variable_ok = relax(FrechetCellFilter(variable), FIRE, fmax=0.05, steps=MAX_STEPS,
                    logfile=str(OUTPUT / 'variable.log'))
for name, atoms, ok in [('fixed', fixed, fixed_ok), ('variable', variable, variable_ok)]:
    print(f'{name}: converged={ok}; energy change={atoms.get_potential_energy()-initial_energy:.4f} eV')
    print(f'Volume change: {100*(atoms.get_volume()/initial.get_volume()-1):.2f}%')
    print('Maximum atomic force:', np.linalg.norm(atoms.get_forces(), axis=1).max(), 'eV/A')
    print('Stress [xx, yy, zz, yz, xz, xy]:', atoms.get_stress(), 'eV/A^3')
    write(OUTPUT / f'{name}.extxyz', atoms)


In [ ]:
view(repair_fragmented_molecules(variable), viewer='x3d')


## Try, explain, and report
1. Did either run hit the step limit? Inspect its log before increasing the limit.
2. Which cell lengths or angles changed? Did molecular connectivity remain plausible?
3. Why can variable-cell relaxation lower energy more than fixed-cell relaxation?
4. Repeat with the other input and compare *changes within each structure*, not raw total energies between them.

**Checkpoint:** record convergence, energy change, volume change, and residual stress.
Model suitability for intermolecular interactions and dispersion must be assessed before interpreting
this as a prediction of crystal stability. An experimental structure can also reflect temperature and pressure
conditions absent from this static relaxation.
